# 00_env_config — shared FabricOps runtime bootstrap

This notebook is loaded by downstream notebooks (`01_dsa`, `02_ex`, `03_pc`) using `%run 00_env_config`.
It initializes shared runtime config, helper functions, AI prompts, and startup validation.


## Shared runtime setup


In [ ]:
ENV = "dev"
AGREEMENT_ID = "replace_me"
SOURCE_LAYER = "source"
TARGET_LAYER = "unified"
AI_ENABLED = True
VALIDATION_MODE = "warn"  # warn for templates; switch to strict for production 03_pc notebooks


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from datetime import datetime, timezone
import uuid
from typing import Any

try:
    spark  # type: ignore[name-defined]
except NameError:
    spark = None


In [ ]:
@dataclass(frozen=True)
class Housepath:
    workspace_id: str
    item_id: str
    item_name: str
    abfss_path: str


@dataclass(frozen=True)
class RuntimeContext:
    env: str
    agreement_id: str
    source_layer: str
    target_layer: str
    run_id: str
    run_timestamp: str
    ai_enabled: bool
    validation_mode: str


In [ ]:
ENV_REGISTRY: dict[str, dict[str, Housepath]] = {
    "dev": {
        "source": Housepath("00000000-0000-0000-0000-000000000001", "11111111-1111-1111-1111-111111111111", "lh_source_dev", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_source_dev"),
        "unified": Housepath("00000000-0000-0000-0000-000000000001", "22222222-2222-2222-2222-222222222222", "lh_unified_dev", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_unified_dev"),
        "product": Housepath("00000000-0000-0000-0000-000000000001", "33333333-3333-3333-3333-333333333333", "lh_product_dev", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_product_dev"),
        "metadata": Housepath("00000000-0000-0000-0000-000000000001", "44444444-4444-4444-4444-444444444444", "lh_metadata_dev", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_metadata_dev"),
        "warehouse": Housepath("00000000-0000-0000-0000-000000000001", "55555555-5555-5555-5555-555555555555", "wh_ops_dev", "sql://warehouse/dev"),
    },
    "prod": {
        "source": Housepath("99999999-9999-9999-9999-999999999991", "aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaa", "lh_source_prod", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_source_prod"),
        "unified": Housepath("99999999-9999-9999-9999-999999999991", "bbbbbbbb-bbbb-bbbb-bbbb-bbbbbbbbbbbb", "lh_unified_prod", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_unified_prod"),
        "product": Housepath("99999999-9999-9999-9999-999999999991", "cccccccc-cccc-cccc-cccc-cccccccccccc", "lh_product_prod", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_product_prod"),
        "metadata": Housepath("99999999-9999-9999-9999-999999999991", "dddddddd-dddd-dddd-dddd-dddddddddddd", "lh_metadata_prod", "abfss://workspace@onelake.dfs.fabric.microsoft.com/lh_metadata_prod"),
        "warehouse": Housepath("99999999-9999-9999-9999-999999999991", "eeeeeeee-eeee-eeee-eeee-eeeeeeeeeeee", "wh_ops_prod", "sql://warehouse/prod"),
    },
}


## Shared helpers


In [ ]:
def validate_environment(env: str) -> None:
    if env not in ENV_REGISTRY:
        raise ValueError(f"Unknown env '{env}'. Available: {sorted(ENV_REGISTRY)}")


def validate_target(target: str, env: str = ENV) -> None:
    validate_environment(env)
    if target not in ENV_REGISTRY[env]:
        raise ValueError(f"Unknown target '{target}' for env '{env}'. Available: {sorted(ENV_REGISTRY[env])}")


def get_path(target: str, env: str = ENV) -> Housepath:
    validate_target(target, env)
    return ENV_REGISTRY[env][target]


def _validate_no_cross_environment_leakage() -> None:
    for env_name, targets in ENV_REGISTRY.items():
        for _, hp in targets.items():
            if env_name not in hp.item_name:
                raise ValueError(f"Expected env marker '{env_name}' in item_name '{hp.item_name}'.")


In [ ]:
def _resolve_spark():
    if spark is None:
        raise RuntimeError("Spark runtime not detected. Use Microsoft Fabric runtime for Spark helpers.")
    return spark


def active_lakehouse_table_read(table_name: str):
    """Read a table from the currently attached active lakehouse."""
    return _resolve_spark().table(table_name)


def active_lakehouse_table_write(df, table_name: str, mode: str = "overwrite") -> None:
    """Write a table to the currently attached active lakehouse."""
    df.write.mode(mode).saveAsTable(table_name)


def lakehouse_path_csv_read(target: str, relative_path: str, env: str = ENV, header: bool = True):
    hp = get_path(target, env)
    return _resolve_spark().read.option("header", str(header).lower()).csv(f"{hp.abfss_path}/{relative_path}")


def lakehouse_path_parquet_read(target: str, relative_path: str, env: str = ENV):
    hp = get_path(target, env)
    return _resolve_spark().read.parquet(f"{hp.abfss_path}/{relative_path}")


def warehouse_read_placeholder(query: str, env: str = ENV):
    """Placeholder: implement workspace-specific Fabric warehouse read integration."""
    hp = get_path("warehouse", env)
    raise RuntimeError(f"warehouse_read_placeholder is not wired. Configure a warehouse connector for {hp.item_name}.")


def warehouse_write_placeholder(df, table_name: str, env: str = ENV):
    """Placeholder: implement workspace-specific Fabric warehouse write integration."""
    hp = get_path("warehouse", env)
    raise RuntimeError(f"warehouse_write_placeholder is not wired. Configure a warehouse connector for {hp.item_name}.")


In [ ]:
METADATA_TABLES = {
    "profile_rows": "metadata.profile_rows",
    "dq_rules": "metadata.dq_rules",
    "lineage_events": "metadata.lineage_events",
    "handover_summaries": "metadata.handover_summaries",
}
OUTPUT_NAMING = {"unified_table_prefix": "unified_", "product_table_prefix": "product_"}
TECHNICAL_COLUMNS = ["run_id", "run_timestamp", "source_system", "record_hash"]
AUDIT_COLUMNS = ["created_at", "created_by", "updated_at", "updated_by"]
RUN_ID = str(uuid.uuid4())
RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()

def clean_datetime_columns(df, columns: list[str]):
    from pyspark.sql import functions as F
    for col in columns:
        df = df.withColumn(col, F.to_timestamp(F.col(col)))
    return df

def add_system_technical_columns(df, run_id: str = RUN_ID, run_timestamp: str = RUN_TIMESTAMP):
    from pyspark.sql import functions as F
    return df.withColumn("run_id", F.lit(run_id)).withColumn("run_timestamp", F.to_timestamp(F.lit(run_timestamp)))


## Shared AI prompts


In [ ]:
AI_PROMPTS = {
    "business_context": """Input: dataset description, ownership, approved use cases, and constraints.
Output: JSON with keys {business_goal, approved_usage, prohibited_usage, key_risks, assumptions}. Keep concise and governance-safe.""",
    "dq_rule_suggestion": """Input: schema, profiling metrics, and sample anomalies.
Output: Markdown table with columns [rule_name, rule_expression, severity, rationale, false_positive_risk]. Rules must be testable.""",
    "dq_rule_review": """Input: candidate DQ rules and business constraints.
Output: JSON array with {rule_name, decision(approve/revise/reject), reason, revision}. Flag non-deterministic rules.""",
    "governance_classification": """Input: column names, sample values (masked), and policy notes.
Output: Markdown table [column, classification, sensitivity_level, pii_flag, rationale, reviewer_questions].""",
    "profile_summary": """Input: profiling outputs including nulls, uniqueness, distributions, and outliers.
Output: 5-bullet summary + 3 prioritized follow-up checks.""",
    "lineage_summary": """Input: source tables, transforms, outputs, and controls.
Output: numbered lineage steps + control checkpoints + unresolved lineage gaps.""",
    "handover_summary": """Input: approved metadata, DQ status, lineage notes, and operational run evidence.
Output: handover note with sections [scope, dependencies, controls, runbook, open actions].""",
}


## Startup validation


In [ ]:
ALLOWED_NOTEBOOK_PREFIXES = ("00_env_config", "01_dsa_", "02_ex_", "03_pc_")


def _detect_active_notebook_name() -> str | None:
    try:
        import notebookutils  # type: ignore
        ctx = notebookutils.runtime.context  # type: ignore[attr-defined]
        for attr in ("notebookName", "name"):
            value = getattr(ctx, attr, None)
            if isinstance(value, str) and value:
                return value
            if callable(value):
                resolved = value()
                if isinstance(resolved, str) and resolved:
                    return resolved
    except Exception:
        return None
    return None


def check_naming_convention(notebook_name: str | None = None) -> bool:
    candidate = notebook_name or _detect_active_notebook_name()
    if not candidate:
        message = "Unable to detect active notebook name. Pass notebook_name explicitly or use Fabric notebook runtime context."
        if VALIDATION_MODE == "strict":
            raise ValueError(message)
        print(f"WARNING: {message}")
        return False
    ok = any(candidate.startswith(prefix) for prefix in ALLOWED_NOTEBOOK_PREFIXES)
    if not ok:
        message = f"Notebook name '{candidate}' does not match allowed prefixes: {ALLOWED_NOTEBOOK_PREFIXES}"
        if VALIDATION_MODE == "strict":
            raise ValueError(message)
        print(f"WARNING: {message}")
    return ok


def validate_runtime() -> None:
    if VALIDATION_MODE == "strict" and spark is None:
        raise RuntimeError("Spark runtime not detected. Strict mode requires Fabric runtime.")


def validate_config() -> None:
    validate_environment(ENV)
    for target in ("source", "unified", "product", "metadata", "warehouse"):
        validate_target(target, ENV)
    if (not AGREEMENT_ID or AGREEMENT_ID == "replace_me") and VALIDATION_MODE == "strict":
        raise ValueError("AGREEMENT_ID is still placeholder in strict mode.")


def validate_paths() -> None:
    _validate_no_cross_environment_leakage()


def validate_ai_ready() -> None:
    required = {"business_context", "dq_rule_suggestion", "dq_rule_review", "governance_classification", "profile_summary", "lineage_summary", "handover_summary"}
    missing = required - set(AI_PROMPTS)
    if AI_ENABLED and missing:
        raise ValueError(f"Missing AI prompt keys: {sorted(missing)}")


def initialize_fabricops_runtime(notebook_name: str | None = None) -> RuntimeContext:
    check_naming_convention(notebook_name)
    validate_runtime()
    validate_config()
    validate_paths()
    validate_ai_ready()
    return RuntimeContext(ENV, AGREEMENT_ID, SOURCE_LAYER, TARGET_LAYER, RUN_ID, RUN_TIMESTAMP, AI_ENABLED, VALIDATION_MODE)


In [ ]:
RUNTIME_CONTEXT = initialize_fabricops_runtime()
print("FabricOps runtime loaded")
print(f"- env: {RUNTIME_CONTEXT.env}")
print(f"- agreement_id: {RUNTIME_CONTEXT.agreement_id}")
print(f"- source target: {get_path(SOURCE_LAYER).item_name}")
print(f"- target layer: {RUNTIME_CONTEXT.target_layer}")
print(f"- metadata target: {get_path('metadata').item_name}")
print(f"- AI enabled: {RUNTIME_CONTEXT.ai_enabled}")
print(f"- validation mode: {RUNTIME_CONTEXT.validation_mode}")
